# Zero-shot Korean IGT with Bolmo 1B

Agenda 17 Nov: two settings, **all-in-one** and **step-by-step** (transliteration -> gloss).
Input = original Korean script + English translation. Output = transliteration + gloss.
Prompts: `zero_shot/prompts/` (Protocol V3, prompt version v2). Logic: `zero_shot/run_zero_shot.py`.

Runtime: **Runtime -> Change runtime type -> GPU** before running.

In [ ]:
!nvidia-smi -L
import sys
print(sys.version)
assert sys.version_info >= (3, 11), "Bolmo needs xlstm, which needs Python >= 3.11"

## 1. Get the code

In [ ]:
import os
REPO = "Korean-IGT-Bolmo"
if not os.path.isdir(REPO):
    !git clone https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git
else:
    !git -C {REPO} pull
%cd {REPO}
!git log --oneline -1

Install dependencies. If transformers was already imported in this session, restart the session after this cell and run again from the top.


In [ ]:
# Bolmo was tested with transformers 4.57.3; Colab ships 5.x, which can break its custom code.
!pip install "transformers==4.57.3" "xlstm==2.0.4" datasets pandas accelerate

import importlib.util
missing = [m for m in ("xlstm", "mlstm_kernels") if importlib.util.find_spec(m) is None]
assert not missing, f"Not installed: {missing} - read the pip output above"
import transformers
print("transformers", transformers.__version__)
if transformers.__version__ != "4.57.3":
    print(">>> Restart the session (Runtime -> Restart session), then run the notebook again from the top.")

## 2. Check the prompts on one dev example (no model needed)

In [ ]:
import sys, json
sys.path.insert(0, "zero_shot")
import importlib, run_zero_shot as rz
importlib.reload(rz)

cfg = json.load(open("configs/zero_shot.json"))
T = {k: rz.load_prompt(v) for k, v in cfg["prompts"].items()}
ex = rz.load_rows("dev", 1)[0]
print(rz.render(T["all_in_one"], original_script=ex["text"], translation=ex["translation"]))
print("-" * 60)
print(rz.render(T["step_1_transliteration"], original_script=ex["text"], translation=ex["translation"]))
print("-" * 60)
print(rz.render(T["step_2_gloss"], original_script=ex["text"], translation=ex["translation"],
                predicted_transliteration="<step-1 prediction goes here>"))
print("-" * 60)
print("gold:", ex["transcription"], "|", ex["glosses"])

## 3. Smoke test: 10 dev examples, both settings

Check that outputs are generated and parsed. This is **not** a reported result.

In [ ]:
import pandas as pd
smoke = rz.main(["--split", "dev", "--limit", "10"])
for p in smoke:
    df = pd.read_csv(p)
    print(p.name, df.parse_status.value_counts().to_dict(), "truncated:", int(df.truncated.sum()))
    display(df[["id", "gold_transliteration", "pred_transliteration", "gold_gloss", "pred_gloss", "parse_status"]])

## 4. Full run on the test split (263 examples)

Only run after the smoke test looks right and the prompts are frozen. Do not change prompts after looking at test outputs.

In [ ]:
full = rz.main(["--split", "test"])
for p in full:
    df = pd.read_csv(p)
    print(p.name, df.parse_status.value_counts().to_dict(), "truncated:", int(df.truncated.sum()))

## 5. Save the results

Download the zip, then put the files in `results/zero_shot/` in the repo and push (or send them to Soyeong for evaluation).

In [ ]:
!zip -r zero_shot_results.zip results/zero_shot
from google.colab import files
files.download("zero_shot_results.zip")